In [0]:
fetch_date = dbutils.widgets.get("fetch_date")
cubeservice_table = dbutils.widgets.get("cubeservice_table")
hchbredarreportvrs_table = dbutils.widgets.get("hchbredarreportvrs_table")
client_table = dbutils.widgets.get("client_table")
agedtrailbalance_report_table = dbutils.widgets.get("agedtrailbalance_report_table")
office_table = dbutils.widgets.get("office_table")
date_table = dbutils.widgets.get("date_table")
masteraging_table = dbutils.widgets.get("masteraging_table")

In [0]:
LastDayInQtr = spark.sql(f"""
SELECT MAX(ServiceDate) AS LastDayInQtr
FROM {cubeservice_table}
WHERE ServiceQuarterNbr = (
    SELECT ServiceQuarterNbr
    FROM {cubeservice_table}
    WHERE ServiceDate = CAST('{fetch_date}' AS DATE)
)
""").collect()[0]['LastDayInQtr']

In [0]:
display(
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW hchb_ma_src AS
SELECT * FROM (
-- CTE 1: Load & Deduplicate HCHBRedARReportVRS
WITH temp_dedup AS (
    SELECT *,
        ROW_NUMBER() OVER (
            PARTITION BY 
                InvNum, ClientName, DOB, HCHB_Company, Client_Balance, EpisodeStart,
                EpisodeEnd, Subscriber_Id, MedicareBeneficiaryIdentifier, FirstSubmissionDate,
                LastPaymentApplied, FirstDOS, LastDOS, Charges, Payments, StillOwe, 
                0_60, 61_90, 91_120, 121_150, 151_180, 181_,
                LastBillingNote, LastBillingNoteDate, LastBillingNoteType, LastBillingNoteCommentType,
                LastFollowupDate, LastBillingNoteUser, PayorType, PayorName, PayorClaimNumber, BranchID,
                Division, Area, Region, State, EpisodeID, BillingFrequency, 181_270, 271_360,
                361_450, 451_540, 541_ , BillDate, PdgmPeriodId
            ORDER BY InvNum, ClientName
        ) AS rnb
    FROM {hchbredarreportvrs_table}
),
temp AS (
    SELECT * 
    FROM temp_dedup 
    WHERE rnb = 1 
    AND Area <> 'AREA: A. MIMS'
),
-- CTE 2: Get latest client data per MedicalRecordNumber
client_latest AS (
    SELECT *
    FROM (
        SELECT 
            MedicalRecordNumber, 
            SystemStatusCode, 
            PayorProgramName, 
            PayorTypeCode, 
            SourceSystem,
            ROW_NUMBER() OVER (
                PARTITION BY MedicalRecordNumber 
                ORDER BY StatusDate DESC
            ) AS rnb
        FROM {client_table}
        WHERE MedicalRecordNumber IS NOT NULL 
        AND SourceSystem = 'HCHB'
    )
    WHERE rnb = 1
),
-- CTE 3: Load base dataset (replaces #temp1)
temp1 AS (
    SELECT 
        atbf.ClientName AS Reimbursement_Team,
        atbf.Facility AS Facility,
        atbf.FacilityCode AS Office_Number,
        ofc.OfficeAbbreviation AS Office_Abbreviation,
        ofc.NationalProviderIdentifier AS NPI,
        ofc.OfficeFEIN AS Tax_ID,
        ofc.Division AS Division,
        atbf.PatientType AS Practice,
        atbf.ServiceType AS State,
        atbf.MedicalRecordNumber AS Client_Number,
        atbf.Patient AS Client_Name,
        try_to_date(atbf.PatientDOB, 'MM/dd/yyyy') AS Client_DOB,
        clt.SystemStatusCode AS Client_Status,
        atbf.AccountNumber AS Invoice_Number,
        atbf.AccountAge AS Age_from_Last_DOS,
        DATEDIFF(DAY, try_to_date(atbf.LastBillDate, 'MM/dd/yyyy'), atbf._reporting_date) AS Age_From_Bill_Date,
        DATEDIFF(DAY, try_to_date(atbf.ClaimThruDate, 'MM/dd/yyyy'), CAST('{LastDayInQtr}' AS DATE)) AS Age_From_End_of_Quarter,
        try_to_date(atbf.LastBillDate, 'MM/dd/yyyy') AS Bill_Date,
        dt.WeekendingDate AS WE_Date,
        try_to_date(atbf.ClaimFromDate, 'MM/dd/yyyy') AS Claim_From_Date,
        try_to_date(atbf.ClaimThruDate, 'MM/dd/yyyy') AS Claim_Through_Date,
        try_to_date(atbf.AdmitDate, 'MM/dd/yyyy') AS Admit_Date,
        try_to_date(atbf.DischargeDate, 'MM/dd/yyyy') AS Discharge_Date,
        redar.EpisodeStart AS Episode_Start,
        redar.EpisodeEnd AS Episode_End,
        try_CAST(redar.EpisodeID AS BIGINT) AS Episode_ID,
        atbf.PayerCategory AS Payer_Category,
        atbf.CurrentFC AS Payer_Type,
        atbf.ActiveInsName AS Payer_Name,
        clt.PayorProgramName AS Program_Name,
        atbf.ActiveInsCode AS Active_Ins_Code_Bill_To,
        NULL AS Billing_Frequency,
        atbf.TotalCharges AS Total_Charges,
        atbf.ExpectedNetRevenue AS Expected_Net_Revenue,
        atbf.TotalAdjustments AS Total_Adjustments,
        atbf.TotalPayments AS Total_Payments,
        atbf.AccountBalance AS Account_Balance,
        CASE 
            WHEN DATEDIFF(DAY, try_to_date(atbf.ClaimThruDate, 'MM/dd/yyyy'), atbf._reporting_date) <= 30 
            THEN COALESCE(CAST(AccountBalance AS DOUBLE), 0.0)  
        END AS 0_30,
        CASE 
            WHEN DATEDIFF(DAY, try_to_date(atbf.ClaimThruDate, 'MM/dd/yyyy'), atbf._reporting_date) BETWEEN 31 AND 60 
            THEN COALESCE(CAST(AccountBalance AS DOUBLE), 0.0)  
        END AS 31_60,
        CASE 
            WHEN DATEDIFF(DAY, try_to_date(atbf.ClaimThruDate, 'MM/dd/yyyy'), atbf._reporting_date) BETWEEN 61 AND 90 
            THEN COALESCE(CAST(AccountBalance AS DOUBLE), 0.0)  
        END AS 61_90,
        CASE 
            WHEN DATEDIFF(DAY, try_to_date(atbf.ClaimThruDate, 'MM/dd/yyyy'), atbf._reporting_date) BETWEEN 91 AND 180 
            THEN COALESCE(CAST(AccountBalance AS DOUBLE), 0.0)  
        END AS 91_180,
        CASE 
            WHEN DATEDIFF(DAY, try_to_date(atbf.ClaimThruDate, 'MM/dd/yyyy'), atbf._reporting_date) BETWEEN 181 AND 270 
            THEN COALESCE(CAST(AccountBalance AS DOUBLE), 0.0)  
        END AS 181_270,
        CASE 
            WHEN DATEDIFF(DAY, try_to_date(atbf.ClaimThruDate, 'MM/dd/yyyy'), atbf._reporting_date) >= 271 
            THEN COALESCE(CAST(AccountBalance AS DOUBLE), 0.0)  
        END AS 271_,
        CASE 
            WHEN DATEDIFF(DAY, try_to_date(atbf.ClaimThruDate, 'MM/dd/yyyy'), CAST('{LastDayInQtr}' AS DATE)) >= 181 
            THEN COALESCE(CAST(AccountBalance AS DOUBLE), 0.0)  
        END AS EOQ_Aged_AR_Impact_181__Days,
        CASE 
            WHEN DATEDIFF(DAY, try_to_date(atbf.ClaimThruDate, 'MM/dd/yyyy'), CAST('{LastDayInQtr}' AS DATE)) >= 271 
            THEN COALESCE(CAST(AccountBalance AS DOUBLE), 0.0)  
        END AS EOQ_Aged_AR_Impact_271__Days,
        CASE 
            WHEN DATEDIFF(DAY, try_to_date(atbf.ClaimThruDate, 'MM/dd/yyyy'), atbf._reporting_date) BETWEEN 271 AND 365 
            THEN COALESCE(CAST(AccountBalance AS DOUBLE), 0.0)  
        END AS 271_365,
        CASE 
            WHEN DATEDIFF(DAY, try_to_date(atbf.ClaimThruDate, 'MM/dd/yyyy'), atbf._reporting_date) BETWEEN 366 AND 540 
            THEN COALESCE(CAST(AccountBalance AS DOUBLE), 0.0)  
        END AS 366_540,
        CASE 
            WHEN DATEDIFF(DAY, try_to_date(atbf.ClaimThruDate, 'MM/dd/yyyy'), atbf._reporting_date) >= 541 
            THEN COALESCE(CAST(AccountBalance AS DOUBLE), 0.0)  
        END AS 541_,
        atbf.ClaimStatus AS Claim_Status,
        atbf.Status AS Account_Status,
        atbf.ContactType AS Contact_Type,
        atbf.AssignedTo AS Collector,
        try_to_date(atbf.LastAction, 'MM/dd/yyyy') AS Last_Action,
        atbf.DaysUntouched AS Untouched,
        try_to_date(atbf.FollowUpDate, 'MM/dd/yyyy') AS Follow_Up_Date,
        try_CAST(atbf.DelinquentDays AS BIGINT) AS Deliquent_Days,
        atbf.LastNote AS Last_User_Note,
        atbf.LastNoteBy AS Last_User_Note_By,
        try_to_date(atbf.LastNoteDate, 'MM/dd/yyyy') AS Last_User_Note_Date,
        atbf.PhysicianName AS Physician_Name,
        atbf.IsHighPriority AS Is_High_Priority,
        atbf.IsDelinquent AS Is_Delinquent,
        try_CAST(atbf.NumOfTouches AS BIGINT) AS Num_Of_Touches,
        atbf.RiskAssessment AS Risk_Assessment,
        'HCHB' AS Source_System,
        atbf._reporting_date AS _reporting_date,
        redar.LastPaymentApplied AS LastPaymentApplied,
        ofc.Region AS Office_Region,
        atbf.ProjectName AS ProjectName,
        atbf.ProjectOwner AS ProjectOwner,
        atbf.TrackingNumberInternal AS TrackingNumberInternal,
        atbf.TrackingNumberExternal AS TrackingNumberExternal,
        atbf.LastProjectNoteDate AS LastProjectNoteDate,
        atbf.LastProjectNoteBy AS LastProjectNoteBy,
        atbf.LastProjectNote AS LastProjectNote
    FROM {agedtrailbalance_report_table} atbf
    INNER JOIN {office_table} ofc
        ON ofc.OfficeNumber = atbf.FacilityCode
    LEFT JOIN temp redar
        ON CAST(redar.InvNum AS STRING) = atbf.AccountNumber
    LEFT JOIN {date_table} dt
        ON try_to_date(atbf.ClaimThruDate, 'MM/dd/yyyy')= dt.CalendarDate
    LEFT JOIN client_latest clt
        ON clt.MedicalRecordNumber = atbf.MedicalRecordNumber
    WHERE atbf.ActiveInsCode NOT LIKE '%-%' 
    AND atbf.AccountNumber NOT LIKE '%ADV%'
    AND atbf._reporting_date = CAST('{fetch_date}' AS DATE)
),
-- CTE 4: Summary balance by Client (Positive)
temp2 AS (
    SELECT 
        MedicalRecordNumber,
        SUM(CAST(AccountBalance AS DOUBLE)) AS Total_Positive_Client_Balance
    FROM {agedtrailbalance_report_table}
    WHERE CAST(AccountBalance AS DOUBLE) > 0
    AND _reporting_date = CAST('{fetch_date}' AS DATE)
    AND ActiveInsCode NOT LIKE '%-%'
    AND AccountNumber NOT LIKE '%ADV%'
    GROUP BY MedicalRecordNumber
),
-- CTE 5: Summary balance by Client (Negative)
temp3 AS (
    SELECT 
        MedicalRecordNumber,
        SUM(CAST(AccountBalance AS DOUBLE)) AS Total_Negative_Client_Balance
    FROM {agedtrailbalance_report_table}
    WHERE CAST(AccountBalance AS DOUBLE) < 0
    AND _reporting_date = CAST('{fetch_date}' AS DATE)
    AND ActiveInsCode NOT LIKE '%-%'
    AND AccountNumber NOT LIKE '%ADV%'
    GROUP BY MedicalRecordNumber
),
-- CTE 6: Apply COALESCE for NULL handling and join aggregates
RawResult AS (
    SELECT 
        t1.Reimbursement_Team AS Reimbursement_Team,
        t1.Facility AS Facility,
        t1.Office_Number AS Office_Number,
        t1.Office_Abbreviation AS Office_Abbreviation,
        t1.NPI AS NPI,
        t1.Tax_ID AS Tax_ID,
        t1.Division AS Division,
        t1.Practice AS Practice,
        t1.State AS State,
        t1.Client_Number AS Client_Number,
        t1.Client_Name AS Client_Name,
        t1.Client_DOB AS Client_DOB,
        t1.Client_Status AS Client_Status,
        t1.Invoice_Number,
        try_CAST(t1.Age_from_Last_DOS AS BIGINT) AS Age_from_Last_DOS,
        try_CAST(t1.Age_From_Bill_Date AS BIGINT) AS Age_From_Bill_Date,
        try_CAST(t1.Age_From_End_of_Quarter AS BIGINT) AS Age_From_End_of_Quarter,
        t1.Bill_Date AS Bill_Date,
        t1.WE_Date AS WE_Date,
        t1.Claim_From_Date AS Claim_From_Date,
        t1.Claim_Through_Date AS Claim_Through_Date,
        t1.Admit_Date AS Admit_Date,
        t1.Discharge_Date AS Discharge_Date,
        DATE_FORMAT(try_to_date(t1.Episode_Start, 'MM/dd/yyyy'),'yyyy-MM-dd') AS Episode_Start,
        DATE_FORMAT(try_to_date(t1.Episode_End, 'MM/dd/yyyy'),'yyyy-MM-dd') AS Episode_End,
        t1.Episode_ID AS Episode_ID,
        t1.Payer_Category AS Payer_Category,
        t1.Payer_Type AS Payer_Type,
        t1.Payer_Name AS Payer_Name,
        t1.Program_Name AS Program_Name,
        t1.Active_Ins_Code_Bill_To AS Active_Ins_Code_Bill_To,
        t1.Billing_Frequency AS Billing_Frequency,
        COALESCE(CAST(t2.Total_Positive_Client_Balance AS DOUBLE), 0.0) AS Total_Positive_Client_Balance,
        COALESCE(CAST(t3.Total_Negative_Client_Balance AS DOUBLE), 0.0) AS Total_Negative_Client_Balance,
        COALESCE(try_CAST(t1.Total_Charges AS DOUBLE), 0.0) AS Total_Charges,
        COALESCE(try_CAST(t1.Expected_Net_Revenue AS DOUBLE), 0.0) AS Expected_Net_Revenue,
        COALESCE(try_CAST(t1.Total_Adjustments AS DOUBLE), 0.0) AS Total_Adjustments,
        COALESCE(try_CAST(t1.Total_Payments AS DOUBLE), 0.0) AS Total_Payments,
        COALESCE(try_CAST(t1.Account_Balance AS DOUBLE), 0.0) AS Account_Balance,
        COALESCE(CAST(t1.0_30 AS DOUBLE), 0.0) AS 0_30,
        COALESCE(CAST(t1.31_60 AS DOUBLE), 0.0) AS 31_60,
        COALESCE(CAST(t1.61_90 AS DOUBLE), 0.0) AS 61_90,
        COALESCE(CAST(t1.91_180 AS DOUBLE), 0.0) AS 91_180,
        COALESCE(CAST(t1.181_270 AS DOUBLE), 0.0) AS 181_270,
        COALESCE(CAST(t1.271_ AS DOUBLE), 0.0) AS 271_,
        COALESCE(CAST(t1.EOQ_Aged_AR_Impact_181__Days AS DOUBLE), 0.0) AS EOQ_Aged_AR_Impact_181__Days,
        COALESCE(CAST(t1.EOQ_Aged_AR_Impact_271__Days AS DOUBLE), 0.0) AS EOQ_Aged_AR_Impact_271__Days,
        COALESCE(CAST(t1.271_365 AS DOUBLE), 0.0) AS 271_365,
        COALESCE(CAST(t1.366_540 AS DOUBLE), 0.0) AS 366_540,
        COALESCE(CAST(t1.541_ AS DOUBLE), 0.0) AS 541_,
        t1.Claim_Status AS Claim_Status,
        t1.Account_Status AS Account_Status,
        t1.Contact_Type AS Contact_Type,
        t1.Collector AS Collector,
        t1.Last_Action AS Last_Action,
        try_CAST(t1.Untouched AS BIGINT) AS Untouched,
        t1.Follow_Up_Date AS Follow_Up_Date,
        t1.Deliquent_Days,
        REGEXP_REPLACE(t1.Last_User_Note, '\\|', '') AS Last_User_Note,
        t1.Last_User_Note_By,
        t1.Last_User_Note_Date,
        t1.Physician_Name,
        t1.Is_High_Priority,
        t1.Is_Delinquent,
        t1.Num_Of_Touches,
        t1.Risk_Assessment,
        t1.Source_System,
        t1._reporting_date,
        DATE_FORMAT(try_to_date(t1.LastPaymentApplied, 'MM/dd/yyyy'),'yyyy-MM-dd') AS Last_Payment_Date,
        t1.Office_Region,
        t1.ProjectName,
        t1.ProjectOwner,
        t1.TrackingNumberInternal,
        t1.TrackingNumberExternal,
        t1.LastProjectNoteDate,
        t1.LastProjectNoteBy,
        t1.LastProjectNote
    FROM temp1 t1
    LEFT JOIN temp2 t2 ON t1.Client_Number = t2.MedicalRecordNumber
    LEFT JOIN temp3 t3 ON t1.Client_Number = t3.MedicalRecordNumber
),
-- CTE 7: Final Deduplication
FinalDeduped AS (
    SELECT *,
        ROW_NUMBER() OVER (
            PARTITION BY 
                Invoice_Number, Client_Number, _reporting_date,
                Account_Balance, Total_Payments, Total_Adjustments, 
                ProjectName, Last_Action, Last_User_Note_Date
            ORDER BY Invoice_Number
        ) AS rn
    FROM RawResult
)
SELECT * FROM FinalDeduped
)
""")
)


In [0]:
display(
spark.sql(f"""
MERGE INTO {masteraging_table} AS target
USING (
    SELECT * FROM hchb_ma_src WHERE rn = 1
) AS source
ON 
    target.Invoice_Number = source.Invoice_Number
    AND target.Client_Number = source.Client_Number
    AND target.reporting_date = source._reporting_date

WHEN MATCHED THEN
    UPDATE SET
        target.Reimbursement_Team = source.Reimbursement_Team,
        target.Facility = source.Facility,
        target.Office_Number = source.Office_Number,
        target.Office_Abbreviation = source.Office_Abbreviation,
        target.NPI = source.NPI,
        target.Tax_ID = source.Tax_ID,
        target.Division = source.Division,
        target.Practice = source.Practice,
        target.State = source.State,
        target.Client_Name = source.Client_Name,
        target.Client_DOB = source.Client_DOB,
        target.Client_Status = source.Client_Status,
        target.Age_from_Last_DOS = source.Age_from_Last_DOS,
        target.Age_From_Bill_Date = source.Age_From_Bill_Date,
        target.Age_From_End_of_Quarter = source.Age_From_End_of_Quarter,
        target.Bill_Date = source.Bill_Date,
        target.week_ending_date = source.WE_Date,
        target.Claim_From_Date = source.Claim_From_Date,
        target.Claim_Through_Date = source.Claim_Through_Date,
        target.Admit_Date = source.Admit_Date,
        target.Discharge_Date = source.Discharge_Date,
        target.Episode_Start = source.Episode_Start,
        target.Episode_End = source.Episode_End,
        target.Episode_ID = source.Episode_ID,
        target.Payer_Category = source.Payer_Category,
        target.Payer_Type = source.Payer_Type,
        target.Payer_Name = source.Payer_Name,
        target.Program_Name = source.Program_Name,
        target.Active_Ins_Code_Bill_To = source.Active_Ins_Code_Bill_To,
        target.Billing_Frequency = source.Billing_Frequency,
        target.Total_Positive_Client_Balance = source.Total_Positive_Client_Balance,
        target.Total_Negative_Client_Balance = source.Total_Negative_Client_Balance,
        target.Total_Charges = source.Total_Charges,
        target.Expected_Net_Revenue = source.Expected_Net_Revenue,
        target.Days_0_30 = source.0_30,
        target.Days_31_60 = source.31_60,
        target.Days_61_90 = source.61_90,
        target.Days_91_180 = source.91_180,
        target.Days_181_270 = source.181_270,
        target.Days_271_Plus = source.271_,
        target.EOQ_Aged_AR_Impact_181_Plus_Days = source.EOQ_Aged_AR_Impact_181__Days,
        target.EOQ_Aged_AR_Impact_271_Plus_Days = source.EOQ_Aged_AR_Impact_271__Days,
        target.Days_271_365 = source.271_365,
        target.Days_366_540 = source.366_540,
        target.Days_541_Plus = source.541_,
        target.Claim_Status = source.Claim_Status,
        target.Account_Status = source.Account_Status,
        target.Contact_Type = source.Contact_Type,
        target.Collector = source.Collector,
        target.Days_Untouched = source.Untouched,
        target.Follow_Up_Date = source.Follow_Up_Date,
        target.Deliquent_Days = source.Deliquent_Days,
        target.Last_User_Note = source.Last_User_Note,
        target.Last_User_Note_By = source.Last_User_Note_By,
        target.Physician_Name = source.Physician_Name,
        target.Is_High_Priority = source.Is_High_Priority,
        target.Is_Delinquent = source.Is_Delinquent,
        target.Num_Of_Touches = source.Num_Of_Touches,
        target.Risk_Assessment = source.Risk_Assessment,
        target.Source_System = source.Source_System,
        target.Last_Payment_Date = source.Last_Payment_Date,
        target.Office_Region = source.Office_Region,
        target.project_owner = source.ProjectOwner,
        target.tracking_number_internal = source.TrackingNumberInternal,
        target.Tracking_Number_External = source.TrackingNumberExternal,
        target.Last_Project_Note_Date = source.LastProjectNoteDate,
        target.Last_Project_Note_By = source.LastProjectNoteBy,
        target.Last_Project_Note = source.LastProjectNote

WHEN NOT MATCHED THEN
    INSERT (
        Reimbursement_Team, Facility, Office_Number, Office_Abbreviation, NPI, Tax_ID, 
        Division, Practice, State, Client_Number, Client_Name, Client_DOB, Client_Status, 
        Invoice_Number, Age_from_Last_DOS, Age_From_Bill_Date, Age_From_End_of_Quarter, 
        Bill_Date, week_ending_date, Claim_From_Date, Claim_Through_Date, Admit_Date, Discharge_Date, 
        Episode_Start, Episode_End, Episode_ID, Payer_Category, Payer_Type, Payer_Name, 
        Program_Name, Active_Ins_Code_Bill_To, Billing_Frequency, Total_Positive_Client_Balance, 
        Total_Negative_Client_Balance, Total_Charges, Expected_Net_Revenue, Total_Adjustments, 
        Total_Payments, Account_Balance, Days_0_30, Days_31_60, Days_61_90, Days_91_180, 
        Days_181_270, Days_271_Plus, EOQ_Aged_AR_Impact_181_Plus_Days, 
        EOQ_Aged_AR_Impact_271_Plus_Days, Days_271_365, Days_366_540, Days_541_Plus, 
        Claim_Status, Account_Status, Contact_Type, Collector, Last_Action, Days_Untouched, 
        Follow_Up_Date, Deliquent_Days, Last_User_Note, Last_User_Note_By, Last_User_Note_Date, 
        Physician_Name, Is_High_Priority, Is_Delinquent, Num_Of_Touches, Risk_Assessment, 
        Source_System, reporting_date, Last_Payment_Date, Office_Region, Project_Name, 
        project_owner, Tracking_Number_Internal, Tracking_Number_External, Last_Project_Note_Date, 
        Last_Project_Note_By, Last_Project_Note
    )
    VALUES (
        source.Reimbursement_Team, source.Facility, source.Office_Number, 
        source.Office_Abbreviation, source.NPI, source.Tax_ID, source.Division, 
        source.Practice, source.State, source.Client_Number, source.Client_Name,
        source.Client_DOB, source.Client_Status, source.Invoice_Number, 
        source.Age_from_Last_DOS, source.Age_From_Bill_Date, source.Age_From_End_of_Quarter,
        source.Bill_Date, source.WE_Date, source.Claim_From_Date, source.Claim_Through_Date,
        source.Admit_Date, source.Discharge_Date, source.Episode_Start, source.Episode_End,
        source.Episode_ID, source.Payer_Category, source.Payer_Type, source.Payer_Name,
        source.Program_Name, source.Active_Ins_Code_Bill_To, source.Billing_Frequency,
        source.Total_Positive_Client_Balance, source.Total_Negative_Client_Balance,
        source.Total_Charges, source.Expected_Net_Revenue, source.Total_Adjustments,
        source.Total_Payments, source.Account_Balance, source.0_30, source.31_60,
        source.61_90, source.91_180, source.181_270, source.271_,
        source.EOQ_Aged_AR_Impact_181__Days, source.EOQ_Aged_AR_Impact_271__Days,
        source.271_365, source.366_540, source.541_, source.Claim_Status,
        source.Account_Status, source.Contact_Type, source.Collector, source.Last_Action,
        source.Untouched, source.Follow_Up_Date, source.Deliquent_Days,
        source.Last_User_Note, source.Last_User_Note_By, source.Last_User_Note_Date,
        source.Physician_Name, source.Is_High_Priority, source.Is_Delinquent,
        source.Num_Of_Touches, source.Risk_Assessment, source.Source_System,
        source._reporting_date, source.Last_Payment_Date, source.Office_Region,
        source.ProjectName, source.ProjectOwner, source.TrackingNumberInternal,
        source.TrackingNumberExternal, source.LastProjectNoteDate, source.LastProjectNoteBy,
        source.LastProjectNote
    );
""")
)